# 03 - Risk Classification: Model Development & Results

Trains and compares Logistic Regression, Random Forest, and Gradient
Boosting on the `Risk_Category` proxy label, then checks the selected
model with 5-fold stratified cross-validation. Reuses the exact functions
in `src/train_risk_model.py` so this notebook and the reusable pipeline
never drift apart.


In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline


In [2]:
from sklearn.model_selection import train_test_split
from src.train_risk_model import (
    load_training_frame, evaluate_algorithms, plot_algorithm_comparison,
    plot_confusion_matrix, plot_feature_importance, run_cross_validation,
)
from src import config

df = load_training_frame()
X = df[config.FEATURE_COLUMNS_NUMERIC + config.FEATURE_COLUMNS_CATEGORICAL]
y = df[config.RISK_TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=config.TEST_SIZE, random_state=config.RANDOM_STATE, stratify=y
)
print(f"Train: {len(X_train):,}  Test: {len(X_test):,}")


Train: 30,024  Test: 7,506


## Algorithm comparison

In [4]:
results, fitted = evaluate_algorithms(X_train, y_train, X_test, y_test)
import pandas as pd
pd.DataFrame(results).T


                     accuracy  precision  recall     f1  train_time_sec
Logistic Regression     72.34      69.60   72.34  70.11            0.26
Random Forest           77.71      76.39   77.71  76.54            0.99
Gradient Boosting       79.74      79.25   79.74  79.34           20.03

## Model selection

Gradient Boosting edges out Random Forest on raw test accuracy in most
runs of this dataset. Random Forest is selected anyway: it trains and
predicts an order of magnitude faster, its `feature_importances_` are
simpler to explain to a non-technical underwriter than a 200-stage
boosted ensemble, and it's less prone to overfitting the noisy proxy
label. That's a deliberate engineering trade-off, not just picking
whichever number is highest.


In [6]:
best_pipe = fitted['Random Forest']
cm = plot_confusion_matrix(best_pipe, X_test, y_test)
cm


array([[4467,  301,    6],
       [ 819,  900,  142],
       [  52,  353,  466]])

In [7]:
importance = plot_feature_importance(best_pipe)
importance


Customer_Age             0.298283
Kids_Driving             0.165213
Household_Income         0.124005
Parent_Yes               0.067822
Car_Year                 0.065563
Vehicle_Age              0.064849
Parent_No                0.057443
Car_Use_Private          0.026531
Car_Use_Commercial       0.024184
Marital_Status_Single    0.008501
dtype: float64

## 5-fold cross-validation (stability check)

In [9]:
cv_results = run_cross_validation(X, y)
pd.DataFrame(cv_results).T


        accuracy  precision  recall
fold_1     77.70      76.60   77.70
fold_2     78.27      77.00   78.27
fold_3     77.64      76.37   77.64
fold_4     77.23      75.96   77.23
fold_5     77.70      76.47   77.70
mean       77.71      76.48   77.71
std         0.33       0.34    0.33

The variance across folds is tight, which suggests the model has learned
a stable, generalizable mapping from features to proxy risk tier rather
than picking up on quirks of one particular train/test split.
